**Structured output**

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

**Pydantic**

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

In [ ]:
from pydantic import BaseModel, Field

# Define a Pydantic model for structured output. model will return the output in the form of this formatted object.
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

In [3]:
model_with_structured_output = model.with_structured_output(Movie)

model_with_structured_output

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B50A52A120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B50A52ABA0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The t

In [ ]:
model.invoke("Provide details about the movie Inception")


# this will return a response from model without structured output because have not pass structured output model to the invoke method. The output will be a string containing details about the movie Inception.

AIMessage(content='**Inception (2010) – Quick Reference Guide**\n\n| Category | Details |\n|----------|---------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16, 2010 (United States) |\n| **Running Time** | 148 minutes (2\u202fh\u202f28\u202fmin) |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Director / Writer** | Christopher Nolan |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Budget** | Approx. **$160\u202fmillion** (including extensive practical effects and location shoots) |\n| **Box‑Office Gross** | $836\u202fmillion worldwide (≈\u202f$292\u202fM domestic, $544\u202fM international) |\n| **MPAA Rating** | PG‑13 (for sequences of violence and action, some language) |\n| **Primary Language** | English (with occasional foreign‑language dialogue) |\n| **Filming Locations** | Los Angeles (CA), Tokyo (Japan), Paris (France), London (UK), Tangier (Morocco), Alberta (Canada), and various studio soundstages.

In [7]:
# pass the structured output model with the invoke method to get the structured output in the form of a Movie object.

model_with_structured_output.invoke("Provide details about the movie Inception")

# this will return a structured output in the form of a Movie object that we defined above. The output will include the title, year, genre, director, and rating of the movie Inception.

Movie(title='Inception', year=2010, genre='Science Fiction, Action', director='Christopher Nolan', rating=8.8)

**Message output alongside parsed structure**

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")



model_with_structured_output = model.with_structured_output(Movie, include_raw=True)

# include_raw=True will include the raw output from the model in the response. The raw output will be a string containing details about the movie Inception.

response = model_with_structured_output.invoke("Provide details about the movie Inception")

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We have a function to get movie details. We need to call functions.Movie with appropriate parameters: title "Inception". Probably need to provide director, genre, rating, year. We\'ll call function.', 'tool_calls': [{'id': 'fc_1c79ae8b-eb71-486e-b821-73c4edd9d7fb', 'function': {'arguments': '{"director":"","genre":"","rating":0,"title":"Inception","year":0}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 171, 'total_tokens': 272, 'completion_time': 0.217376714, 'completion_tokens_details': {'reasoning_tokens': 50}, 'prompt_time': 0.006451087, 'prompt_tokens_details': None, 'queue_time': 0.049141623, 'total_time': 0.223827801}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'gr

**Nested Structure**

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    """An actor with details"""
    name: str = Field(description="The name of the actor")
    role: str = Field(description="The role of the actor in the movie")

# we can use the Actor model inside the Movie details model to represent the cast of the movie. The cast will be a list of Actor objects.
class MovieDetails(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    cast: list[Actor] = Field(description="The cast of the movie")
    genre: str = Field(description="The genre of the movie")
    budget: float = Field(None, description="The budget of the movie")


model_with_structured_output = model.with_structured_output(MovieDetails)


response = model_with_structured_output.invoke("Provide details about the movie Inception")

response


# Output
"""
MovieDetails(
    title='Inception', 
    year=2010, 
    cast=[
        Actor(name='Leonardo DiCaprio', role='Dom Cobb'), 
        Actor(name='Joseph Gordon-Levitt', role='Arthur'), 
        Actor(name='Elliot Page', role='Ariadne'), 
        Actor(name='Tom Hardy', role='Eames'), 
        Actor(name='Ken Watanabe', role='Saito'), 
        Actor(name='Marion Cotillard', role='Mal'), 
        Actor(name='Michael Caine', role='Professor Stephen Miles'), 
        Actor(name='Cillian Murphy', role='Robert Fischer')
    ], 
    genre='Science fiction / Thriller', 
    budget=160000000.0
)
"""


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer')], genre='Science fiction / Thriller', budget=160000000.0)

#### TypeDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [14]:
from typing_extensions import TypedDict
from typing import Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_with_typed_dict_output = model.with_structured_output(MovieDict)


model_with_typed_dict_output.invoke("Provide details about the movie Interstellar")

{'director': 'Christopher Nolan',
 'rating': 8.6,
 'title': 'Interstellar',
 'year': 2014}

**Nested Structure with typedict**

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated

class Actor(TypedDict):
    """An actor with details"""
    name: Annotated[str, ..., "The name of the actor"]
    role: Annotated[str, ..., "The role of the actor in the movie"]

class MovieDetails(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    cast: Annotated[list[Actor], ..., "The cast of the movie"]
    genre: Annotated[str, ..., "The genre of the movie"]
    budget: Annotated[float, None, "The budget of the movie"]


model_with_typed_dict_output = model.with_structured_output(MovieDetails)

response = model_with_typed_dict_output.invoke("Provide details about the movie Interstellar")

response

{'budget': 165000000,
 'cast': [{'name': 'Matthew McConaughey', 'role': 'Cooper'},
  {'name': 'Anne Hathaway', 'role': 'Dr. Amelia Brand'},
  {'name': 'Jessica Chastain', 'role': 'Murph (adult)'},
  {'name': 'Mackenzie Foy', 'role': 'Murph (young)'},
  {'name': 'Michael Caine', 'role': 'Professor Brand'},
  {'name': 'Casey Affleck', 'role': 'Tom'},
  {'name': 'David Gyasi', 'role': 'Romilly'},
  {'name': 'Matt Damon', 'role': 'Dr. Mann'},
  {'name': 'Bill Irwin', 'role': 'TARS (voice)'},
  {'name': 'Josh Stewart', 'role': 'CASE (voice)'}],
 'genre': 'Science Fiction, Adventure, Drama',
 'title': 'Interstellar',
 'year': 2014}

In [17]:
model.profile

# this will return a profile of the model including the model name, version, and other details. The profile will be a dictionary containing the model's metadata.

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

#### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [20]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='47c566c9-286e-4401-872e-b772f72ff4dd'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', 'extras': {'signature': 'El4KXAERTTIPaFaRvkXjp6oGVPJ8BhFwc0xa6Ry+OoWDEh9Zca7OPRo1Qak9Awaaq7Bx9lYkOUSbLMWDqPgwQD6T/GR3W0wFSHVJYTqnEJDhUEc7QadCN4+qYROWWMIv'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0807b-b748-78f3-a363-528f014878dd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [21]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [22]:
## Using Typedict

from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## Using Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')